# Fleet Baseline EDA

Reproducible exploratory data analysis for `data/fleet_baseline.parquet`.

In [1]:
from __future__ import annotations
from pathlib import Path as _NotebookPath
import sys as _notebook_sys

for _candidate in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents):
    _notebook_path = _candidate / 'ml_models/00_eda/eda_fleet_baseline.ipynb'
    if _notebook_path.exists():
        __file__ = str(_notebook_path.resolve())
        break
else:
    __file__ = str((_NotebookPath.cwd() / 'ml_models/00_eda/eda_fleet_baseline.ipynb').resolve())

_notebook_sys.argv = [__file__]


## Configuration

Set `DATA_PATH`, `OUT_DIR_OVERRIDE`, or `SKIP_PLOTS` here before executing the notebook.

In [2]:
DATA_PATH = None
OUT_DIR_OVERRIDE = None
SKIP_PLOTS = False

## Implementation

In [3]:
import argparse
import sys
from pathlib import Path
from typing import Any

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError(f"Could not find repository root from {start}")


REPO_ROOT = find_repo_root(Path(__file__).resolve().parent)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sdg.schema import CITY_CATEGORIES, CLIMATE_CATEGORIES, COMPONENT_IDS, STATUS_CATEGORIES


DEFAULT_DATA_PATH = REPO_ROOT / "data" / "fleet_baseline.parquet"
DEFAULT_OUT_DIR = Path(__file__).resolve().parent
FIGURE_DIR_NAME = "figures"

CLIMATE_COLORS = {
    "nordic": "#4477aa",
    "continental": "#66ccee",
    "oceanic": "#228833",
    "mediterranean": "#cc6677",
    "eastern": "#aa3377",
}


def main() -> None:
    args = parse_args()
    data_path = resolve_path(args.data)
    out_dir = resolve_path(args.out_dir)
    figures_dir = out_dir / FIGURE_DIR_NAME

    out_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    metadata = pq.read_metadata(data_path)
    schema = pq.read_schema(data_path)
    df = pd.read_parquet(data_path)
    df["date"] = pd.to_datetime(df["date"])

    summaries = build_summaries(df, metadata, schema, data_path)
    figure_paths: list[Path] = []
    if not args.skip_plots:
        figure_paths = write_figures(summaries, figures_dir)

    report = render_report(summaries, data_path, figure_paths, out_dir)
    report_path = out_dir / "REPORT.md"
    report_path.write_text(report, encoding="utf-8")

    print(f"Wrote {relative_path(report_path)}")
    if figure_paths:
        print(f"Wrote {len(figure_paths)} figures under {relative_path(figures_dir)}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Generate a deterministic EDA report for data/fleet_baseline.parquet."
    )
    parser.add_argument(
        "--data",
        default=str(DEFAULT_DATA_PATH.relative_to(REPO_ROOT)),
        help="Path to the fleet baseline Parquet file. Relative paths are resolved from repo root.",
    )
    parser.add_argument(
        "--out-dir",
        default=str(DEFAULT_OUT_DIR.relative_to(REPO_ROOT)),
        help="Directory for REPORT.md and figures/. Relative paths are resolved from repo root.",
    )
    parser.add_argument(
        "--skip-plots",
        action="store_true",
        help="Only write REPORT.md; skip regenerating PNG figures.",
    )
    return parser.parse_args()


def resolve_path(path_like: str | Path) -> Path:
    path = Path(path_like)
    if path.is_absolute():
        return path
    return REPO_ROOT / path


def build_summaries(
    df: pd.DataFrame,
    metadata: pq.FileMetaData,
    schema: Any,
    data_path: Path,
) -> dict[str, Any]:
    components = list(COMPONENT_IDS)
    status_categories = list(STATUS_CATEGORIES)
    city_categories = list(CITY_CATEGORIES)
    climate_categories = list(CLIMATE_CATEGORIES)
    n_rows = len(df)
    n_printers = int(df["printer_id"].nunique())
    n_days = int(df["date"].nunique())
    date_start = df["date"].min()
    date_end = df["date"].max()
    rows_per_printer = df.groupby("printer_id", observed=True).size()

    dataset_summary = [
        {"metric": "data_file", "value": relative_path(data_path)},
        {"metric": "file_size_mb", "value": f"{data_path.stat().st_size / 1024 / 1024:.2f}"},
        {"metric": "parquet_rows", "value": f"{metadata.num_rows:,}"},
        {"metric": "parquet_columns", "value": metadata.num_columns},
        {"metric": "parquet_row_groups", "value": metadata.num_row_groups},
        {"metric": "dataframe_shape", "value": f"{n_rows:,} x {df.shape[1]:,}"},
        {"metric": "memory_mb_loaded", "value": f"{df.memory_usage(deep=True).sum() / 1024 / 1024:.2f}"},
        {"metric": "date_start", "value": date_start.date().isoformat()},
        {"metric": "date_end", "value": date_end.date().isoformat()},
        {"metric": "calendar_days", "value": f"{n_days:,}"},
        {"metric": "printers", "value": n_printers},
        {"metric": "cities", "value": int(df["city"].nunique())},
        {"metric": "climate_zones", "value": int(df["climate_zone"].nunique())},
        {"metric": "rows_per_printer_min", "value": f"{int(rows_per_printer.min()):,}"},
        {"metric": "rows_per_printer_max", "value": f"{int(rows_per_printer.max()):,}"},
        {"metric": "components", "value": ", ".join(components)},
    ]

    schema_groups = pd.DataFrame(
        [
            {"group": "identity", "columns": "printer_id, city, climate_zone"},
            {"group": "calendar", "columns": "date, day"},
            {
                "group": "weather_and_load",
                "columns": "ambient_temp_c, humidity_pct, dust_concentration, Q_demand, daily_print_hours, cumulative_print_hours",
            },
            {"group": "health", "columns": ", ".join(f"H_{component}" for component in components)},
            {
                "group": "status",
                "columns": ", ".join(f"status_{component}" for component in components),
            },
            {
                "group": "maintenance_clock",
                "columns": ", ".join(f"tau_{component}" for component in components),
            },
            {
                "group": "age_since_replacement",
                "columns": ", ".join(f"L_{component}" for component in components),
            },
            {"group": "counters", "columns": "N_f, N_c, N_TC, N_on"},
            {
                "group": "hazard_rate",
                "columns": ", ".join(f"lambda_{component}" for component in components),
            },
            {
                "group": "maintenance_events",
                "columns": ", ".join(f"maint_{component}" for component in components),
            },
            {
                "group": "failure_events",
                "columns": ", ".join(f"failure_{component}" for component in components),
            },
            {
                "group": "rul_labels",
                "columns": ", ".join([*(f"rul_{component}" for component in components), "rul_system"]),
            },
        ]
    )

    city_summary = summarize_cities(df, city_categories)
    climate_summary = summarize_climates(df, climate_categories)
    environment_summary = summarize_environment(df, climate_categories)
    status_counts, status_share_matrix = summarize_status(df, components, status_categories)
    health_summary = summarize_health(df, components)
    dynamics_summary = summarize_dynamics(df, components)
    event_summary = summarize_events(df, components, n_rows)
    rul_summary = summarize_rul(df, components, n_rows)
    sanity_checks = run_sanity_checks(df, metadata, schema, components, status_categories, n_rows, n_days, n_printers)

    return {
        "components": components,
        "status_categories": status_categories,
        "city_categories": city_categories,
        "climate_categories": climate_categories,
        "dataset_summary": pd.DataFrame(dataset_summary),
        "schema_groups": schema_groups,
        "city_summary": city_summary,
        "climate_summary": climate_summary,
        "environment_summary": environment_summary,
        "status_counts": status_counts,
        "status_share_matrix": status_share_matrix,
        "health_summary": health_summary,
        "dynamics_summary": dynamics_summary,
        "event_summary": event_summary,
        "rul_summary": rul_summary,
        "sanity_checks": sanity_checks,
        "n_rows": n_rows,
        "n_days": n_days,
        "n_printers": n_printers,
        "date_start": date_start,
        "date_end": date_end,
    }


def summarize_cities(df: pd.DataFrame, city_categories: list[str]) -> pd.DataFrame:
    printer_city = (
        df.sort_values(["printer_id", "date"])
        .groupby("printer_id", observed=True)
        .agg(city=("city", "first"), climate_zone=("climate_zone", "first"))
        .reset_index()
    )
    city_printers = (
        printer_city.groupby(["city", "climate_zone"], observed=True)
        .agg(printers=("printer_id", "nunique"))
        .reset_index()
    )
    city_rows = (
        df.groupby(["city", "climate_zone"], observed=True)
        .size()
        .rename("rows")
        .reset_index()
    )
    city_summary = city_printers.merge(city_rows, on=["city", "climate_zone"], how="left")
    city_summary["row_share_pct"] = 100.0 * city_summary["rows"] / len(df)
    city_summary["city_order"] = city_summary["city"].astype(str).map({city: i for i, city in enumerate(city_categories)})
    city_summary = city_summary.sort_values("city_order").drop(columns="city_order")
    return city_summary


def summarize_climates(df: pd.DataFrame, climate_categories: list[str]) -> pd.DataFrame:
    printer_city = (
        df.groupby("printer_id", observed=True)
        .agg(city=("city", "first"), climate_zone=("climate_zone", "first"))
        .reset_index()
    )
    climate_printers = (
        printer_city.groupby("climate_zone", observed=True)
        .agg(cities=("city", "nunique"), printers=("printer_id", "nunique"))
        .reset_index()
    )
    climate_rows = df.groupby("climate_zone", observed=True).size().rename("rows").reset_index()
    climate_summary = climate_printers.merge(climate_rows, on="climate_zone", how="left")
    climate_summary["row_share_pct"] = 100.0 * climate_summary["rows"] / len(df)
    climate_summary["climate_order"] = climate_summary["climate_zone"].astype(str).map(
        {climate: i for i, climate in enumerate(climate_categories)}
    )
    climate_summary = climate_summary.sort_values("climate_order").drop(columns="climate_order")
    return climate_summary


def summarize_environment(df: pd.DataFrame, climate_categories: list[str]) -> pd.DataFrame:
    summary = (
        df.groupby("climate_zone", observed=True)
        .agg(
            ambient_temp_c_mean=("ambient_temp_c", "mean"),
            ambient_temp_c_p05=("ambient_temp_c", lambda series: series.quantile(0.05)),
            ambient_temp_c_p95=("ambient_temp_c", lambda series: series.quantile(0.95)),
            humidity_pct_mean=("humidity_pct", "mean"),
            dust_concentration_mean=("dust_concentration", "mean"),
            q_demand_mean=("Q_demand", "mean"),
            daily_print_hours_mean=("daily_print_hours", "mean"),
        )
        .reset_index()
    )
    summary["climate_order"] = summary["climate_zone"].astype(str).map(
        {climate: i for i, climate in enumerate(climate_categories)}
    )
    return summary.sort_values("climate_order").drop(columns="climate_order")


def summarize_status(
    df: pd.DataFrame,
    components: list[str],
    status_categories: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows: list[dict[str, Any]] = []
    for component in components:
        counts = df[f"status_{component}"].value_counts(dropna=False).reindex(status_categories, fill_value=0)
        for status, count in counts.items():
            rows.append(
                {
                    "component": component,
                    "status": status,
                    "count": int(count),
                    "share_pct": 100.0 * int(count) / len(df),
                }
            )

    status_counts = pd.DataFrame(rows)
    status_share_matrix = (
        status_counts.pivot(index="component", columns="status", values="share_pct")
        .reindex(index=components, columns=status_categories)
        .fillna(0.0)
    )
    return status_counts, status_share_matrix


def summarize_health(df: pd.DataFrame, components: list[str]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for component in components:
        series = df[f"H_{component}"]
        rows.append(
            {
                "component": component,
                "min": series.min(),
                "p01": series.quantile(0.01),
                "p05": series.quantile(0.05),
                "median": series.median(),
                "mean": series.mean(),
                "p95": series.quantile(0.95),
                "max": series.max(),
            }
        )
    return pd.DataFrame(rows)


def summarize_dynamics(df: pd.DataFrame, components: list[str]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for component in components:
        tau = df[f"tau_{component}"]
        age = df[f"L_{component}"]
        hazard = df[f"lambda_{component}"]
        rows.append(
            {
                "component": component,
                "tau_median_h": tau.median(),
                "tau_max_h": tau.max(),
                "L_median_h": age.median(),
                "L_max_h": age.max(),
                "lambda_median_per_h": hazard.median(),
                "lambda_p95_per_h": hazard.quantile(0.95),
                "lambda_max_per_h": hazard.max(),
            }
        )
    return pd.DataFrame(rows)


def summarize_events(df: pd.DataFrame, components: list[str], n_rows: int) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    by_printer = df.groupby("printer_id", observed=True)
    for component in components:
        maint_col = f"maint_{component}"
        failure_col = f"failure_{component}"
        maint_events = int(df[maint_col].sum())
        failure_events = int(df[failure_col].sum())
        rows.append(
            {
                "component": component,
                "maintenance_events": maint_events,
                "failure_events": failure_events,
                "printers_with_maintenance": int(by_printer[maint_col].any().sum()),
                "printers_with_failure": int(by_printer[failure_col].any().sum()),
                "maintenance_row_pct": 100.0 * maint_events / n_rows,
                "failure_row_pct": 100.0 * failure_events / n_rows,
            }
        )
    return pd.DataFrame(rows)


def summarize_rul(df: pd.DataFrame, components: list[str], n_rows: int) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for column in [*(f"rul_{component}" for component in components), "rul_system"]:
        series = df[column]
        observed = series.dropna()
        rows.append(
            {
                "label": column,
                "non_null_rows": int(series.notna().sum()),
                "coverage_pct": 100.0 * series.notna().mean(),
                "zero_rows": int((series == 0).sum()),
                "censored_rows": int(n_rows - series.notna().sum()),
                "min_days": nullable_stat(observed, "min"),
                "median_days": nullable_stat(observed, "median"),
                "p95_days": nullable_quantile(observed, 0.95),
                "max_days": nullable_stat(observed, "max"),
            }
        )
    return pd.DataFrame(rows)


def nullable_stat(series: pd.Series, stat: str) -> float | int | None:
    if series.empty:
        return None
    value = getattr(series, stat)()
    if pd.isna(value):
        return None
    return value


def nullable_quantile(series: pd.Series, q: float) -> float | None:
    if series.empty:
        return None
    value = series.quantile(q)
    if pd.isna(value):
        return None
    return float(value)


def run_sanity_checks(
    df: pd.DataFrame,
    metadata: pq.FileMetaData,
    schema: Any,
    components: list[str],
    status_categories: list[str],
    n_rows: int,
    n_days: int,
    n_printers: int,
) -> pd.DataFrame:
    checks: list[dict[str, str]] = []

    def add(name: str, passed: bool, detail: str) -> None:
        checks.append({"check": name, "status": "PASS" if passed else "FAIL", "detail": detail})

    add(
        "parquet_metadata_matches_loaded_frame",
        metadata.num_rows == n_rows and metadata.num_columns == df.shape[1],
        f"metadata={metadata.num_rows:,}x{metadata.num_columns}, loaded={n_rows:,}x{df.shape[1]}",
    )

    expected_rows = n_printers * n_days
    add(
        "full_printer_day_grid",
        n_rows == expected_rows,
        f"rows={n_rows:,}, printers={n_printers}, days={n_days:,}, expected={expected_rows:,}",
    )

    duplicates = int(df.duplicated(["printer_id", "date"]).sum())
    add("no_duplicate_printer_dates", duplicates == 0, f"duplicate printer/date rows={duplicates:,}")

    rows_per_printer = df.groupby("printer_id", observed=True).size()
    add(
        "each_printer_has_same_day_count",
        bool((rows_per_printer == n_days).all()),
        f"min={int(rows_per_printer.min()):,}, max={int(rows_per_printer.max()):,}, expected={n_days:,}",
    )

    date_start = df["date"].min()
    expected_dates = date_start + pd.to_timedelta(df["day"].astype("int64"), unit="D")
    mismatched_dates = int((df["date"] != expected_dates).sum())
    add(
        "day_column_matches_date_offset",
        mismatched_dates == 0,
        f"rows where date != first_date + day: {mismatched_dates:,}",
    )

    printer_city_counts = df.groupby("printer_id", observed=True)[["city", "climate_zone"]].nunique()
    unstable_printers = int(((printer_city_counts["city"] != 1) | (printer_city_counts["climate_zone"] != 1)).sum())
    add(
        "printer_city_and_climate_are_static",
        unstable_printers == 0,
        f"printers with changing city or climate={unstable_printers:,}",
    )

    city_climate_counts = df.groupby("city", observed=True)["climate_zone"].nunique()
    drifting_cities = int((city_climate_counts != 1).sum())
    add("each_city_maps_to_one_climate", drifting_cities == 0, f"cities with multiple climates={drifting_cities:,}")

    health_cols = [f"H_{component}" for component in components]
    health_min = float(df[health_cols].min().min())
    health_max = float(df[health_cols].max().max())
    add(
        "health_values_in_unit_interval",
        health_min >= 0.0 and health_max <= 1.0,
        f"global health min={health_min:.6f}, max={health_max:.6f}",
    )

    invalid_status: list[str] = []
    expected_status_set = set(status_categories)
    for component in components:
        observed = set(df[f"status_{component}"].dropna().astype(str).unique())
        unknown = sorted(observed - expected_status_set)
        if unknown:
            invalid_status.append(f"{component}: {', '.join(unknown)}")
    add(
        "status_values_are_known_categories",
        not invalid_status,
        "all component status columns are within " + ", ".join(status_categories)
        if not invalid_status
        else "; ".join(invalid_status),
    )

    event_nulls = 0
    event_non_bool = []
    for prefix in ("maint", "failure"):
        for component in components:
            column = f"{prefix}_{component}"
            event_nulls += int(df[column].isna().sum())
            if str(df[column].dtype) != "bool":
                event_non_bool.append(f"{column}:{df[column].dtype}")
    add(
        "event_columns_are_boolean_without_nulls",
        event_nulls == 0 and not event_non_bool,
        f"nulls={event_nulls:,}" if not event_non_bool else f"non_bool={'; '.join(event_non_bool)}",
    )

    failure_rul_mismatches = 0
    rul_negative_rows = 0
    for component in components:
        failure_col = f"failure_{component}"
        rul_col = f"rul_{component}"
        failure_rul_mismatches += int((df.loc[df[failure_col], rul_col] != 0).sum())
        rul_negative_rows += int((df[rul_col].dropna() < 0).sum())
    add(
        "failure_rows_have_zero_component_rul",
        failure_rul_mismatches == 0,
        f"component failure rows with nonzero RUL={failure_rul_mismatches:,}",
    )
    add("component_rul_is_nonnegative", rul_negative_rows == 0, f"negative component RUL rows={rul_negative_rows:,}")

    rul_cols = [f"rul_{component}" for component in components]
    expected_system = df[rul_cols].min(axis=1, skipna=True)
    expected_system = expected_system.mask(df[rul_cols].isna().all(axis=1))
    lhs = expected_system.astype("Float64").fillna(-1).astype("int64")
    rhs = df["rul_system"].astype("Float64").fillna(-1).astype("int64")
    system_mismatches = int((lhs != rhs).sum())
    add(
        "system_rul_is_min_observed_component_rul",
        system_mismatches == 0,
        f"rows where rul_system disagrees with component minimum={system_mismatches:,}",
    )

    schema_names = list(schema.names)
    add(
        "schema_contains_expected_final_columns",
        all(column in schema_names for column in [*health_cols, *rul_cols, "rul_system"]),
        f"schema columns={len(schema_names)}",
    )

    return pd.DataFrame(checks)


def write_figures(summaries: dict[str, Any], figures_dir: Path) -> list[Path]:
    plt.rcParams.update(
        {
            "figure.dpi": 140,
            "savefig.dpi": 140,
            "font.size": 9,
            "axes.titlesize": 11,
            "axes.labelsize": 9,
            "xtick.labelsize": 8,
            "ytick.labelsize": 8,
        }
    )

    paths = [
        plot_city_distribution(summaries["city_summary"], figures_dir),
        plot_climate_distribution(summaries["climate_summary"], figures_dir),
        plot_status_heatmap(summaries["status_share_matrix"], figures_dir),
        plot_health_boxplot(summaries["health_summary"], figures_dir),
        plot_events(summaries["event_summary"], figures_dir),
        plot_rul_coverage(summaries["rul_summary"], figures_dir),
    ]
    return paths


def plot_city_distribution(city_summary: pd.DataFrame, figures_dir: Path) -> Path:
    path = figures_dir / "city_printer_distribution.png"
    fig, ax = plt.subplots(figsize=(10, 4.8))
    colors = [CLIMATE_COLORS.get(str(zone), "#888888") for zone in city_summary["climate_zone"]]
    ax.bar(city_summary["city"].astype(str), city_summary["printers"], color=colors, edgecolor="#333333", linewidth=0.5)
    ax.set_title("Printers per city")
    ax.set_ylabel("printer count")
    ax.set_xlabel("city")
    ax.set_ylim(0, max(city_summary["printers"]) + 1)
    ax.tick_params(axis="x", rotation=45)
    handles = [
        plt.Line2D([0], [0], marker="s", linestyle="", color=color, label=climate)
        for climate, color in CLIMATE_COLORS.items()
        if climate in set(city_summary["climate_zone"].astype(str))
    ]
    ax.legend(handles=handles, title="climate", frameon=False, ncols=3, loc="upper right")
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_climate_distribution(climate_summary: pd.DataFrame, figures_dir: Path) -> Path:
    path = figures_dir / "climate_row_distribution.png"
    fig, ax = plt.subplots(figsize=(7, 4.4))
    zones = climate_summary["climate_zone"].astype(str)
    colors = [CLIMATE_COLORS.get(zone, "#888888") for zone in zones]
    ax.bar(zones, climate_summary["rows"], color=colors, edgecolor="#333333", linewidth=0.5)
    ax.set_title("Rows per climate zone")
    ax.set_ylabel("daily rows")
    ax.set_xlabel("climate zone")
    ax.ticklabel_format(axis="y", style="plain")
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_status_heatmap(status_share_matrix: pd.DataFrame, figures_dir: Path) -> Path:
    path = figures_dir / "component_status_share.png"
    values = status_share_matrix.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(7.2, 4.3))
    image = ax.imshow(values, cmap="YlGnBu", aspect="auto", vmin=0, vmax=max(100.0, float(values.max())))
    ax.set_title("Component status distribution")
    ax.set_xlabel("status")
    ax.set_ylabel("component")
    ax.set_xticks(np.arange(status_share_matrix.shape[1]), labels=status_share_matrix.columns)
    ax.set_yticks(np.arange(status_share_matrix.shape[0]), labels=status_share_matrix.index)
    threshold = float(values.max()) / 2.0 if values.size else 0.0
    for row_idx in range(values.shape[0]):
        for col_idx in range(values.shape[1]):
            value = values[row_idx, col_idx]
            color = "white" if value > threshold else "#222222"
            ax.text(col_idx, row_idx, f"{value:.1f}%", ha="center", va="center", color=color, fontsize=8)
    fig.colorbar(image, ax=ax, label="row share (%)")
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_health_boxplot(health_summary: pd.DataFrame, figures_dir: Path) -> Path:
    path = figures_dir / "component_health_summary.png"
    fig, ax = plt.subplots(figsize=(8, 4.8))
    components = health_summary["component"].tolist()
    box_stats = []
    for row in health_summary.to_dict("records"):
        box_stats.append(
            {
                "label": row["component"],
                "whislo": row["min"],
                "q1": row["p05"],
                "med": row["median"],
                "q3": row["p95"],
                "whishi": row["max"],
                "fliers": [],
            }
        )
    ax.bxp(box_stats, showfliers=False, patch_artist=True, boxprops={"facecolor": "#88ccee", "edgecolor": "#333333"})
    ax.set_title("Component health distribution")
    ax.set_ylabel("health H")
    ax.set_xlabel("component")
    ax.set_ylim(-0.02, 1.02)
    ax.set_xticks(range(1, len(components) + 1), labels=components)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_events(event_summary: pd.DataFrame, figures_dir: Path) -> Path:
    path = figures_dir / "maintenance_failure_events.png"
    fig, ax = plt.subplots(figsize=(8, 4.8))
    x = np.arange(len(event_summary))
    width = 0.36
    ax.bar(x - width / 2, event_summary["maintenance_events"], width, label="maintenance", color="#44aa99")
    ax.bar(x + width / 2, event_summary["failure_events"], width, label="failure", color="#cc6677")
    ax.set_title("Maintenance and failure events by component")
    ax.set_ylabel("event count")
    ax.set_xlabel("component")
    ax.set_xticks(x, labels=event_summary["component"])
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_rul_coverage(rul_summary: pd.DataFrame, figures_dir: Path) -> Path:
    path = figures_dir / "rul_label_coverage.png"
    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    labels = rul_summary["label"]
    colors = ["#117733" if label == "rul_system" else "#88ccee" for label in labels]
    ax.bar(labels, rul_summary["coverage_pct"], color=colors, edgecolor="#333333", linewidth=0.5)
    ax.set_title("RUL label coverage")
    ax.set_ylabel("non-null rows (%)")
    ax.set_xlabel("label")
    ax.set_ylim(0, 105)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def render_report(
    summaries: dict[str, Any],
    data_path: Path,
    figure_paths: list[Path],
    out_dir: Path,
) -> str:
    dataset_summary = summaries["dataset_summary"]
    city_summary = summaries["city_summary"]
    climate_summary = summaries["climate_summary"]
    environment_summary = summaries["environment_summary"]
    status_share_matrix = summaries["status_share_matrix"].reset_index()
    health_summary = summaries["health_summary"]
    dynamics_summary = summaries["dynamics_summary"]
    event_summary = summaries["event_summary"]
    rul_summary = summaries["rul_summary"]
    sanity_checks = summaries["sanity_checks"]

    sections = [
        "# Fleet Baseline EDA",
        "",
        f"Input: `{relative_path(data_path)}`.",
        "",
        "This report profiles the deterministic SDG fleet baseline: one daily row per printer, city, and component state. The SDG generator seeds each printer by `printer_id`, assigns printers to 15 configured European cities, simulates weather and demand, evolves six component health channels `C1`-`C6`, emits maintenance and failure booleans, then derives RUL labels from future failure events.",
        "",
        "A null RUL value means the row is right-censored for that label: no later failure for that component was observed inside the 2015-01-01 to 2024-12-31 dataset horizon.",
        "",
        "## Dataset Shape",
        "",
        markdown_table(dataset_summary),
        "",
        "## Column Groups",
        "",
        markdown_table(summaries["schema_groups"]),
        "",
        "## City and Climate Coverage",
        "",
        markdown_table(
            format_dataframe(
                city_summary,
                {
                    "rows": comma_int,
                    "row_share_pct": pct,
                },
            )
        ),
        "",
        markdown_table(
            format_dataframe(
                climate_summary,
                {
                    "rows": comma_int,
                    "row_share_pct": pct,
                },
            )
        ),
        "",
        "## Weather and Demand by Climate",
        "",
        markdown_table(
            format_dataframe(
                environment_summary,
                {
                    "ambient_temp_c_mean": fixed_2,
                    "ambient_temp_c_p05": fixed_2,
                    "ambient_temp_c_p95": fixed_2,
                    "humidity_pct_mean": fixed_2,
                    "dust_concentration_mean": fixed_2,
                    "q_demand_mean": fixed_3,
                    "daily_print_hours_mean": fixed_3,
                },
            )
        ),
        "",
        "## Component Status Distribution",
        "",
        "Statuses are written after the simulator applies preventive and corrective actions for the day. Because corrective failure handling resets the component before the row is emitted, `status_FAILED` can be zero even when `failure_*` event booleans are true.",
        "",
        markdown_table(format_dataframe(status_share_matrix, {column: pct for column in status_share_matrix.columns if column != "component"})),
        "",
        "## Component Health",
        "",
        markdown_table(
            format_dataframe(
                health_summary,
                {
                    "min": fixed_4,
                    "p01": fixed_4,
                    "p05": fixed_4,
                    "median": fixed_4,
                    "mean": fixed_4,
                    "p95": fixed_4,
                    "max": fixed_4,
                },
            )
        ),
        "",
        "## Tau, L, and Lambda",
        "",
        "`tau_*` is the component maintenance clock in hours, `L_*` is age since replacement in hours, and `lambda_*` is the simulated hazard rate per hour.",
        "",
        markdown_table(
            format_dataframe(
                dynamics_summary,
                {
                    "tau_median_h": fixed_1,
                    "tau_max_h": fixed_1,
                    "L_median_h": fixed_1,
                    "L_max_h": fixed_1,
                    "lambda_median_per_h": sci_3,
                    "lambda_p95_per_h": sci_3,
                    "lambda_max_per_h": sci_3,
                },
            )
        ),
        "",
        "## Maintenance and Failure Events",
        "",
        "The event columns are daily booleans. Counts below are row counts where that same-day event fired.",
        "",
        markdown_table(
            format_dataframe(
                event_summary,
                {
                    "maintenance_events": comma_int,
                    "failure_events": comma_int,
                    "maintenance_row_pct": pct,
                    "failure_row_pct": pct,
                },
            )
        ),
        "",
        "## RUL Label Availability",
        "",
        markdown_table(
            format_dataframe(
                rul_summary,
                {
                    "non_null_rows": comma_int,
                    "coverage_pct": pct,
                    "zero_rows": comma_int,
                    "censored_rows": comma_int,
                    "min_days": nullable_int,
                    "median_days": nullable_1,
                    "p95_days": nullable_1,
                    "max_days": nullable_int,
                },
            )
        ),
        "",
        "## Sanity Checks",
        "",
        markdown_table(sanity_checks),
        "",
    ]

    if figure_paths:
        sections.extend(["## Figures", ""])
        for figure_path in figure_paths:
            sections.extend([f"![{figure_path.stem}]({relative_path(figure_path, out_dir)})", ""])

    sections.extend(
        [
            "## Reproduce",
            "",
            "```bash",
            "uv run jupyter nbconvert --to notebook --execute --inplace ml_models/00_eda/eda_fleet_baseline.ipynb",
            "```",
            "",
            "To point the EDA at another compatible SDG output, edit the notebook configuration cell.",
            "",
        ]
    )
    return "\n".join(sections)


def format_dataframe(df: pd.DataFrame, formatters: dict[str, Any]) -> pd.DataFrame:
    formatted = df.copy()
    for column, formatter in formatters.items():
        if column in formatted.columns:
            formatted[column] = formatted[column].map(formatter)
    return formatted


def markdown_table(df: pd.DataFrame) -> str:
    if df.empty:
        return "_No rows._"

    headers = [str(column) for column in df.columns]
    rows = [[format_cell(value) for value in record] for record in df.to_numpy(dtype=object)]
    widths = [
        max(len(headers[index]), *(len(row[index]) for row in rows))
        for index in range(len(headers))
    ]

    header_line = "| " + " | ".join(headers[index].ljust(widths[index]) for index in range(len(headers))) + " |"
    sep_line = "| " + " | ".join("-" * widths[index] for index in range(len(headers))) + " |"
    body_lines = [
        "| " + " | ".join(row[index].ljust(widths[index]) for index in range(len(headers))) + " |"
        for row in rows
    ]
    return "\n".join([header_line, sep_line, *body_lines])


def format_cell(value: Any) -> str:
    if value is None:
        return ""
    if pd.isna(value):
        return ""
    if isinstance(value, (np.integer, int)):
        return str(int(value))
    if isinstance(value, (np.floating, float)):
        return f"{float(value):.4g}"
    return str(value)


def comma_int(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    return f"{int(value):,}"


def nullable_int(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    return f"{int(round(float(value))):,}"


def nullable_1(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    return f"{float(value):.1f}"


def pct(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    return f"{float(value):.2f}%"


def fixed_1(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    return f"{float(value):.1f}"


def fixed_2(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    return f"{float(value):.2f}"


def fixed_3(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    return f"{float(value):.3f}"


def fixed_4(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    return f"{float(value):.4f}"


def sci_3(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    return f"{float(value):.3e}"


def relative_path(path: Path, root: Path = REPO_ROOT) -> str:
    try:
        return str(path.resolve().relative_to(root.resolve()))
    except ValueError:
        return str(path)



In [4]:
_BASE_BUILD_SUMMARIES = build_summaries
_BASE_WRITE_FIGURES = write_figures
_BASE_RENDER_REPORT = render_report


def summarize_correlations(df: pd.DataFrame) -> pd.DataFrame:
    numeric_columns = [
        column
        for column in df.columns
        if column != "printer_id"
        and (pd.api.types.is_numeric_dtype(df[column]) or pd.api.types.is_bool_dtype(df[column]))
    ]
    correlation_matrix = df[numeric_columns].corr(numeric_only=True).fillna(0.0)
    return correlation_matrix


def _normalize_heatmap_frame(frame: pd.DataFrame) -> pd.DataFrame:
    minimum = frame.min()
    span = (frame.max() - minimum).replace(0, 1.0)
    return (frame - minimum) / span


def _heatmap_text_color(value: float) -> str:
    return "white" if value > 0.65 else "#1f1f1f"


def plot_variable_correlation_heatmap(correlation_matrix: pd.DataFrame, figures_dir: Path) -> Path:
    path = figures_dir / "variable_correlation_heatmap.png"
    values = correlation_matrix.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(18, 16))
    image = ax.imshow(values, cmap="coolwarm", vmin=-1.0, vmax=1.0, aspect="auto")
    ax.set_title("Pearson correlation across numeric variables")
    ax.set_xlabel("variable")
    ax.set_ylabel("variable")
    ax.set_xticks(np.arange(correlation_matrix.shape[1]), labels=correlation_matrix.columns)
    ax.set_yticks(np.arange(correlation_matrix.shape[0]), labels=correlation_matrix.index)
    ax.tick_params(axis="x", rotation=90)
    ax.tick_params(axis="y", labelsize=7)
    fig.colorbar(image, ax=ax, label="correlation")
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_climate_profile_heatmap(environment_summary: pd.DataFrame, figures_dir: Path) -> Path:
    path = figures_dir / "climate_profile_heatmap.png"
    metric_columns = [
        "ambient_temp_c_mean",
        "humidity_pct_mean",
        "dust_concentration_mean",
        "q_demand_mean",
        "daily_print_hours_mean",
    ]
    display_frame = environment_summary.set_index("climate_zone")[metric_columns]
    normalized_frame = _normalize_heatmap_frame(display_frame)
    fig, ax = plt.subplots(figsize=(10, 5.2))
    image = ax.imshow(normalized_frame.to_numpy(dtype=float), cmap="YlGnBu", aspect="auto", vmin=0.0, vmax=1.0)
    ax.set_title("Climate profile across environment and load metrics")
    ax.set_xlabel("metric")
    ax.set_ylabel("climate zone")
    ax.set_xticks(np.arange(len(metric_columns)), labels=[
        "temp",
        "humidity",
        "dust",
        "Q demand",
        "print hrs",
    ])
    ax.set_yticks(np.arange(len(display_frame.index)), labels=display_frame.index)
    for row_idx, climate_zone in enumerate(display_frame.index):
        for col_idx, metric in enumerate(metric_columns):
            value = display_frame.iloc[row_idx, col_idx]
            ax.text(
                col_idx,
                row_idx,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color=_heatmap_text_color(float(normalized_frame.iloc[row_idx, col_idx])),
            )
    fig.colorbar(image, ax=ax, label="normalized within metric")
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_component_dynamics_heatmap(dynamics_summary: pd.DataFrame, figures_dir: Path) -> Path:
    path = figures_dir / "component_dynamics_heatmap.png"
    display_frame = dynamics_summary.set_index("component")[
        ["tau_median_h", "L_median_h", "lambda_median_per_h"]
    ].copy()
    display_frame["lambda_median_per_h"] = np.log10(display_frame["lambda_median_per_h"].clip(lower=np.finfo(float).tiny))
    normalized_frame = _normalize_heatmap_frame(display_frame)
    fig, ax = plt.subplots(figsize=(8.5, 5.6))
    image = ax.imshow(normalized_frame.to_numpy(dtype=float), cmap="magma", aspect="auto", vmin=0.0, vmax=1.0)
    ax.set_title("Component dynamics overview")
    ax.set_xlabel("metric")
    ax.set_ylabel("component")
    ax.set_xticks(np.arange(display_frame.shape[1]), labels=["tau median", "L median", "log10(lambda)"])
    ax.set_yticks(np.arange(display_frame.shape[0]), labels=display_frame.index)
    for row_idx, component in enumerate(display_frame.index):
        tau_value = dynamics_summary.loc[dynamics_summary["component"] == component, "tau_median_h"].iloc[0]
        age_value = dynamics_summary.loc[dynamics_summary["component"] == component, "L_median_h"].iloc[0]
        lambda_value = dynamics_summary.loc[dynamics_summary["component"] == component, "lambda_median_per_h"].iloc[0]
        display_values = [tau_value, age_value, lambda_value]
        annotations = [f"{tau_value:.1f}", f"{age_value:.1f}", f"{lambda_value:.2e}"]
        for col_idx, (annotation, normalized_value) in enumerate(zip(annotations, normalized_frame.iloc[row_idx])):
            ax.text(
                col_idx,
                row_idx,
                annotation,
                ha="center",
                va="center",
                fontsize=8,
                color=_heatmap_text_color(float(normalized_value)),
            )
    fig.colorbar(image, ax=ax, label="normalized within metric")
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_component_risk_profile(
    health_summary: pd.DataFrame,
    event_summary: pd.DataFrame,
    dynamics_summary: pd.DataFrame,
    figures_dir: Path,
) -> Path:
    path = figures_dir / "component_risk_profile.png"
    profile = (
        health_summary[["component", "mean"]]
        .merge(
            event_summary[["component", "maintenance_events", "failure_events", "maintenance_row_pct", "failure_row_pct"]],
            on="component",
            how="inner",
        )
        .merge(dynamics_summary[["component", "lambda_median_per_h"]], on="component", how="inner")
    )
    bubble_size = 80 + 920 * profile["maintenance_events"] / max(float(profile["maintenance_events"].max()), 1.0)
    bubble_color = np.log10(profile["lambda_median_per_h"].clip(lower=np.finfo(float).tiny))
    fig, ax = plt.subplots(figsize=(8.8, 6.2))
    scatter = ax.scatter(
        profile["mean"],
        profile["failure_row_pct"],
        s=bubble_size,
        c=bubble_color,
        cmap="viridis",
        alpha=0.9,
        edgecolors="#1f1f1f",
        linewidths=0.7,
    )
    for row in profile.itertuples(index=False):
        ax.annotate(
            row.component,
            (row.mean, row.failure_row_pct),
            textcoords="offset points",
            xytext=(6, 4),
            fontsize=8,
        )
    ax.set_title("Component health versus failure intensity")
    ax.set_xlabel("mean health")
    ax.set_ylabel("failure rows (%)")
    ax.grid(alpha=0.2)
    colorbar = fig.colorbar(scatter, ax=ax)
    colorbar.set_label("log10(median hazard)")
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def build_summaries(
    df: pd.DataFrame,
    metadata: pq.FileMetaData,
    schema: Any,
    data_path: Path,
) -> dict[str, Any]:
    summaries = _BASE_BUILD_SUMMARIES(df, metadata, schema, data_path)
    summaries["correlation_matrix"] = summarize_correlations(df)
    return summaries


def write_figures(summaries: dict[str, Any], figures_dir: Path) -> list[Path]:
    figure_paths = list(_BASE_WRITE_FIGURES(summaries, figures_dir))
    figure_paths.extend(
        [
            plot_variable_correlation_heatmap(summaries["correlation_matrix"], figures_dir),
            plot_climate_profile_heatmap(summaries["environment_summary"], figures_dir),
            plot_component_dynamics_heatmap(summaries["dynamics_summary"], figures_dir),
            plot_component_risk_profile(
                summaries["health_summary"],
                summaries["event_summary"],
                summaries["dynamics_summary"],
                figures_dir,
            ),
        ]
    )
    return figure_paths


def render_report(
    summaries: dict[str, Any],
    data_path: Path,
    figure_paths: list[Path],
    out_dir: Path,
) -> str:
    report = _BASE_RENDER_REPORT(summaries, data_path, figure_paths, out_dir)
    extra_sections = [
        "## Variable Correlations",
        "",
        "The heatmap below shows Pearson correlation across the numeric variables in the fleet baseline. `printer_id` is excluded because it is an identifier rather than a signal feature.",
        "",
        "## Climate and Load Profile",
        "",
        "This heatmap normalizes each metric column independently so the climate zones can be compared on the same color scale.",
        "",
        "## Component Dynamics Profile",
        "",
        "`tau_*`, `L_*`, and the median hazard rate summarize the component lifecycle state in one compact view.",
        "",
        "## Component Risk Profile",
        "",
        "Bubble size reflects maintenance volume, while color reflects the median hazard rate on a log scale.",
        "",
    ]
    marker = "## Figures\n\n"
    if marker in report:
        return report.replace(marker, "\n".join(extra_sections) + "\n\n" + marker, 1)
    return report + "\n\n" + "\n".join(extra_sections)


In [5]:
_PREVIOUS_BUILD_SUMMARIES = build_summaries
_PREVIOUS_WRITE_FIGURES = write_figures
_PREVIOUS_RENDER_REPORT = render_report

# Targets baked into sdg/config/components.yaml (= L_nom_d per component).
# Used to overlay reference lines on the MTTE chart so it's obvious at a
# glance whether the empirical calibration is on-target.
MTTE_TARGETS_DAYS = {"C1": 33.0, "C2": 750.0, "C3": 50.0, "C4": 208.0, "C5": 500.0, "C6": 2500.0}


def summarize_mean_time_to_error(df: pd.DataFrame, components: list[str]) -> pd.DataFrame:
    """Mean time to *first* failure per component, in DAYS, across the fleet.

    For each (printer, component) we take the day index of the first time
    failure_<C> flips True. MTTE is the mean of those first-failure days
    across printers; std/median/p05/p95 give the spread. Printers that
    never fail within the simulated horizon are excluded.

    Note: this replaces an earlier implementation that averaged the running
    `hours_since_*_failure` counter, which is *not* MTTE — it's the mean
    elapsed time since the most recent failure across the whole timeline.
    """
    rows: list[dict[str, Any]] = []
    for component in components:
        fail_col = f"failure_{component}"
        first_fail = (
            df[df[fail_col] == True]
            .groupby("printer_id")["day"]
            .min()
        )
        n_failed = int(first_fail.shape[0])
        target = MTTE_TARGETS_DAYS.get(component, float("nan"))
        if n_failed == 0:
            rows.append({
                "component": component,
                "target_days": target,
                "mean_days": float("nan"),
                "std_days": float("nan"),
                "median_days": float("nan"),
                "p05_days": float("nan"),
                "p95_days": float("nan"),
                "n_failed": 0,
                "deviation_pct": float("nan"),
            })
            continue
        mean_days = float(first_fail.mean())
        rows.append({
            "component": component,
            "target_days": target,
            "mean_days": mean_days,
            "std_days": float(first_fail.std()),
            "median_days": float(first_fail.median()),
            "p05_days": float(first_fail.quantile(0.05)),
            "p95_days": float(first_fail.quantile(0.95)),
            "n_failed": n_failed,
            "deviation_pct": 100.0 * (mean_days - target) / target if target == target else float("nan"),
        })
    return pd.DataFrame(rows)


def plot_mean_time_to_error(mean_time_to_error: pd.DataFrame, figures_dir: Path) -> Path:
    path = figures_dir / "mean_time_to_error_by_component.png"
    df_plot = mean_time_to_error.copy().sort_values("target_days").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(9.6, 5.0))
    x = np.arange(len(df_plot))
    width = 0.38
    ax.bar(x - width / 2, df_plot["target_days"], width=width, color="#a0c4d4",
           edgecolor="#1f4e62", linewidth=0.6, label="target (L_nom_d)")
    ax.bar(x + width / 2, df_plot["mean_days"], width=width, color="#dd8452",
           edgecolor="#5a3a1f", linewidth=0.6, label="empirical mean")
    ax.errorbar(x + width / 2, df_plot["mean_days"], yerr=df_plot["std_days"],
                fmt="none", ecolor="#5a3a1f", capsize=4, alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(df_plot["component"])
    ax.set_yscale("log")
    ax.set_ylabel("days to first failure (log scale)")
    ax.set_title("Mean time to first failure by component (target vs empirical, +/-1 sigma)")
    ax.grid(axis="y", which="both", alpha=0.25)
    ax.legend(loc="upper left")
    for xi, mean, std, dev in zip(
        x, df_plot["mean_days"], df_plot["std_days"], df_plot["deviation_pct"]
    ):
        if mean != mean:
            continue
        label = f"{mean:.0f}d +/-{std:.0f}\n({dev:+.1f}% vs target)"
        ax.text(xi + width / 2, mean, label, ha="center", va="bottom", fontsize=7.5)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def build_summaries(
    df: pd.DataFrame,
    metadata: pq.FileMetaData,
    schema: Any,
    data_path: Path,
) -> dict[str, Any]:
    summaries = _PREVIOUS_BUILD_SUMMARIES(df, metadata, schema, data_path)
    summaries["mean_time_to_error"] = summarize_mean_time_to_error(df, summaries["components"])
    return summaries


def write_figures(summaries: dict[str, Any], figures_dir: Path) -> list[Path]:
    figure_paths = list(_PREVIOUS_WRITE_FIGURES(summaries, figures_dir))
    figure_paths.append(plot_mean_time_to_error(summaries["mean_time_to_error"], figures_dir))
    return figure_paths


def render_report(
    summaries: dict[str, Any],
    data_path: Path,
    figure_paths: list[Path],
    out_dir: Path,
) -> str:
    report = _PREVIOUS_RENDER_REPORT(summaries, data_path, figure_paths, out_dir)
    mtte_df = summaries["mean_time_to_error"]
    table_lines = [
        "| Component | Target (d) | Empirical mean (d) | +/-1 sigma (d) | Deviation | n failed |",
        "|---|---:|---:|---:|---:|---:|",
    ]
    for _, row in mtte_df.iterrows():
        table_lines.append(
            f"| {row['component']} | {row['target_days']:.0f} | "
            f"{row['mean_days']:.1f} | {row['std_days']:.1f} | "
            f"{row['deviation_pct']:+.1f}% | {int(row['n_failed'])}/100 |"
        )
    extra_section = [
        "## Mean Time to First Failure (days)",
        "",
        "Mean and +/-1 sigma of the day-index when each component first fails, computed across the fleet (one observation per printer per component). With the empirical calibration in `sdg/config/components.yaml` and `alpha_sigma=0.05`, the per-printer MTTE distribution is approximately Normal(L_nom_d, 5%*L_nom_d).",
        "",
        *table_lines,
        "",
    ]
    marker = "## Figures\n\n"
    if marker in report:
        return report.replace(marker, "\n".join(extra_section) + "\n" + marker, 1)
    return report + "\n\n" + "\n".join(extra_section)


## Generate Report

In [6]:
data_path = resolve_path(DATA_PATH if DATA_PATH is not None else DEFAULT_DATA_PATH.relative_to(REPO_ROOT))
out_dir = resolve_path(OUT_DIR_OVERRIDE if OUT_DIR_OVERRIDE is not None else DEFAULT_OUT_DIR.relative_to(REPO_ROOT))
figures_dir = out_dir / FIGURE_DIR_NAME

out_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

metadata = pq.read_metadata(data_path)
schema = pq.read_schema(data_path)
df = pd.read_parquet(data_path)
df["date"] = pd.to_datetime(df["date"])

summaries = build_summaries(df, metadata, schema, data_path)
figure_paths: list[Path] = []
if not SKIP_PLOTS:
    figure_paths = write_figures(summaries, figures_dir)

report = render_report(summaries, data_path, figure_paths, out_dir)
report_path = out_dir / "REPORT.md"
report_path.write_text(report, encoding="utf-8")

print(f"Wrote {relative_path(report_path)}")
if figure_paths:
    print(f"Wrote {len(figure_paths)} figures under {relative_path(figures_dir)}")


Wrote ml_models/00_eda/REPORT.md
Wrote 11 figures under ml_models/00_eda/figures
